### Dataset 

| Case | Generative model | Mean | Covariance |
|-----:|------------------|------|------------|
| **Uncorrelated (i.i.d.)** | $ y_i \sim \mathcal N(0,\sigma^2),\; i=1,\dots,n $ | $ \mathbb E[y]=0 $ | $ \mathrm{Cov}(y)=\sigma^2 I $ |
| **Correlated (GP)** | $ y = L\varepsilon,\; \varepsilon\sim\mathcal N(0,I) $ | $ \mathbb E[y]=0 $ | $ \mathrm{Cov}(y)=K+\sigma_n^2 I \equiv K_y $ |

In [1]:
import numpy as np

def generate_datasets(
    *,
    n=40,
    x_min=0.0,
    x_max=10.0,
    seed=7,
    sigma_iid=1.0,          # iid noise std
    kernel_fn=None,         # kernel_fn(x, x) -> (n,n) covariance
    sigma_corr_noise=0.2,   # measurement noise std (diagonal)
    jitter=1e-10,           # tiny diagonal stabilizer
):
    if kernel_fn is None:
        raise ValueError("kernel_fn must be provided (callable returning an (n,n) covariance matrix).")

    rng = np.random.default_rng(seed)

    # Inputs
    x = np.linspace(x_min, x_max, n)

    # Dataset A: i.i.d.
    y_uncorr = rng.normal(0.0, sigma_iid, size=n)

    # Dataset B: correlated via Cholesky
    K = np.asarray(kernel_fn(x, x), dtype=np.float64)
    if K.shape != (n, n):
        raise ValueError(f"kernel_fn returned shape {K.shape}, expected {(n, n)}")

    # Defensive symmetrization
    K = 0.5 * (K + K.T)

    # Observed covariance
    K_y = K + (sigma_corr_noise**2) * np.eye(n)
    K_y += jitter * np.eye(n)

    # Sample
    L = np.linalg.cholesky(K_y)
    eps = rng.normal(0.0, 1.0, size=n)
    y_corr = L @ eps

    return x, y_uncorr, y_corr, K_y

## Kernel Squared Exponential (SE)

$$
k(x_i, x_j)
=
\sigma_f^2
\exp\!\left(
-\frac{(x_i - x_j)^2}{2\ell^2}
\right),
$$

$$
K_{ij} = k(x_i, x_j).
$$

In [2]:
# -------------------------
# Squared Exponential Decay / Radial Basis Function
# -------------------------
def rbf_kernel(x1, x2, sigma_f=1.5, ell=1.2):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    sqdist = (x1 - x2) ** 2
    return (sigma_f**2) * np.exp(-0.5 * sqdist / (ell**2))


## Other Kernels

In [3]:
# -------------------------
# Linear kernel
# k(x, x') = sigma_f^2 (x - c)(x' - c)
# -------------------------
def linear_kernel(x1, x2, sigma_f=1.0, c=0.0):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    return sigma_f**2 * (x1 - c) * (x2 - c)


# -------------------------
# Polynomial kernel
# -------------------------
def polynomial_kernel(x1, x2, sigma_f=1.0, degree=2, c=1.0):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    return sigma_f**2 * (x1 @ x2.T + c) ** degree


# -------------------------
# White noise kernel
# k(x, x') = sigma_n^2 δ(x - x')
# -------------------------
def white_kernel(x1, x2, sigma_n=1.0):
    x1 = np.asarray(x1)
    x2 = np.asarray(x2)
    if len(x1) != len(x2):
        raise ValueError("White kernel requires x1 and x2 of same length")
    return sigma_n**2 * np.eye(len(x1))


## Plot

In [4]:
import ROOT
import numpy as np
import uuid

def plot_datasets_root_inline(
    x,
    y_uncorr,
    y_corr,
    *,
    title_left="Dataset A: i.i.d.",
    title_right="Dataset B: correlated",
    canvas_name="c",
    canvas_title="Correlated vs Non-correlated",
    width=1100,
    height=450,
):
    # ROOT config: inline
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    # Avoid canvas name collisions
    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"

    c = ROOT.TCanvas(cname, canvas_title, width, height)
    c.Divide(2, 1)

    # Keep references alive (%jsroot)
    c._graphs = []

    def make_graph(xarr, yarr, name):
        g = ROOT.TGraph(
            len(xarr),
            np.asarray(xarr, dtype=np.float64),
            np.asarray(yarr, dtype=np.float64),
        )
        g.SetName(f"{name}_{uid}")
        g.SetMarkerStyle(20)
        g.SetMarkerSize(1.0)
        return g

    # -------- Left: i.i.d. --------
    c.cd(1)
    g1 = make_graph(x, y_uncorr, "g_uncorr")
    g1.SetTitle(f"{title_left};x;y")
    g1.Draw("AP")
    c._graphs.append(g1)

    # -------- Right: correlated --------
    c.cd(2)
    g2 = make_graph(x, y_corr, "g_corr")
    g2.SetTitle(f"{title_right};x;y")
    g2.Draw("AP")
    c._graphs.append(g2)

    c.Modified()
    c.Update()

    # JSROOT inline render
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)

    return c

/opt/homebrew/Cellar/root/6.38.00/lib/root/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /Users/chanayo/.pyenv/versions/3.14.0/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


## Experiment

In [6]:
# Dataset parameters
sigma_iid_real = 0.5
sigma_corr_real = 0
sigma_f_real = 1.5
ell_real=1.2

# generate kernel
def kernel_fn(a, b):
    return rbf_kernel(a, b, sigma_f_real, ell_real)

# generate dataset
x, y_uncorr, y_corr, K_y = generate_datasets(
    sigma_iid=sigma_iid_real,
    kernel_fn=kernel_fn,
    sigma_corr_noise=sigma_corr_real,
)

# plot
c = plot_datasets_root_inline(x, y_uncorr, y_corr)


In [7]:
import numpy as np
import scipy
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize

# -----------------------------
# RBF / SE kernel matrix
# -----------------------------
def K_rbf(x, sigma_f, ell):
    x = np.asarray(x, dtype=np.float64).reshape(-1, 1)
    sqdist = (x - x.T) ** 2
    return (sigma_f**2) * np.exp(-0.5 * sqdist / (ell**2))

# -----------------------------
# Log marginal likelihood (and negative)
# Params are in log-space for stability:
# theta = [log_sigma_f, log_ell, log_sigma_n]
# -----------------------------
def nll_gp_rbf(theta, x, y, jitter=1e-10):
    log_sigma_f, log_ell, log_sigma_n = theta
    sigma_f = np.exp(log_sigma_f)
    ell     = np.exp(log_ell)
    sigma_n = np.exp(log_sigma_n)

    y = np.asarray(y, dtype=np.float64).reshape(-1)
    n = len(y)

    K = K_rbf(x, sigma_f, ell)
    Ky = K + (sigma_n**2 + jitter) * np.eye(n)

    # Cholesky
    c, lower = cho_factor(Ky, lower=True, check_finite=False)
    alpha = cho_solve((c, lower), y, check_finite=False)

    # log |Ky| = 2 * sum(log(diag(L)))
    logdet = 2.0 * np.sum(np.log(np.diag(c)))

    # log p(y|theta) = -0.5 y^T Ky^{-1} y -0.5 logdet - n/2 log(2pi)
    lml = -0.5 * y.dot(alpha) - 0.5 * logdet - 0.5 * n * np.log(2*np.pi)

    return -lml  # negative for minimization

# -----------------------------
# Fit helper
# -----------------------------
def fit_gp_rbf_hyperparams(x, y, *, init=None, bounds=None, jitter=1e-10):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    # Heuristic init if not provided
    if init is None:
        ystd = np.std(y) if np.std(y) > 0 else 1.0
        init_sigma_f = ystd
        init_ell     = 1.0
        init_sigma_n = 0.3 * ystd + 1e-6
        init = np.log([init_sigma_f, init_ell, init_sigma_n])

    # Reasonable bounds in log-space
    if bounds is None:
        # sigma_f in [1e-6, 1e3], ell in [1e-3, 1e3], sigma_n in [1e-8, 1e3]
        bounds = [(-14, 7), (-7, 7), (-18, 7)]

    obj = lambda th: nll_gp_rbf(th, x, y, jitter=jitter)

    res = minimize(
        obj,
        x0=np.asarray(init, dtype=np.float64),
        method="L-BFGS-B",
        bounds=bounds,
    )

    log_sigma_f, log_ell, log_sigma_n = res.x
    sigma_f = float(np.exp(log_sigma_f))
    ell     = float(np.exp(log_ell))
    sigma_n = float(np.exp(log_sigma_n))

    # Return also LML at optimum
    lml = -nll_gp_rbf(res.x, x, y, jitter=jitter)

    out = {
        "sigma_f": sigma_f,
        "ell": ell,
        "sigma_n": sigma_n,
        "log_marginal_likelihood": float(lml),
        "success": bool(res.success),
        "message": res.message,
        "nfev": res.nfev,
    }
    return out

# -----------------------------
# Run fits on both datasets
# -----------------------------
fit_uncorr = fit_gp_rbf_hyperparams(x, y_uncorr)
fit_corr   = fit_gp_rbf_hyperparams(x, y_corr)

print("=== GP fit: Dataset A (uncorrelated / i.i.d.) ===")
for k,v in fit_uncorr.items():
    print(f"{k}: {v}")

print("\n=== GP fit: Dataset B (correlated) ===")
for k,v in fit_corr.items():
    print(f"{k}: {v}")

=== GP fit: Dataset A (uncorrelated / i.i.d.) ===
sigma_f: 0.1921225710848941
ell: 2.9562434157849533
sigma_n: 0.39744957496106204
log_marginal_likelihood: -21.897335994797828
success: True
message: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
nfev: 80

=== GP fit: Dataset B (correlated) ===
sigma_f: 0.7217320870223193
ell: 1.1238850221521133
sigma_n: 1.1529416875340892e-05
log_marginal_likelihood: 199.75786292132642
success: True
message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
nfev: 156


In [8]:
import sys
print(sys.executable)

/Users/chanayo/Library/Mobile Documents/com~apple~CloudDocs/Repository/Academia/Course-Codex/cern-root-student-course/.venv/bin/python


In [9]:
import pandas as pd
import numpy as np

def make_real_vs_learned_table(
    *,
    sigma_iid_real,
    sigma_corr_real,
    sigma_f_real,
    ell_real,
    fit_uncorr,
    fit_corr,
):
    # "Real" parameters depend on the dataset construction:
    # - Dataset A (i.i.d.): y ~ N(0, sigma_iid_real^2 I)
    #   -> no correlated signal, so sigma_f_real_A = 0, ell_real_A = NaN (not defined)
    #   -> sigma_n_real_A = sigma_iid_real
    #
    # - Dataset B (correlated): y ~ N(0, K_SE(sigma_f_real, ell_real) + sigma_corr_real^2 I)
    #   -> sigma_f_real_B = sigma_f_real
    #   -> ell_real_B = ell_real
    #   -> sigma_n_real_B = sigma_corr_real

    rows = []

    # Dataset A
    rows.append({
        "dataset": "A (i.i.d.)",
        "param": "sigma_f",
        "real": 0.0,
        "learned": fit_uncorr["sigma_f"],
        "abs_err": abs(fit_uncorr["sigma_f"] - 0.0),
        "rel_err": np.nan,  # undefined relative error vs 0
    })
    rows.append({
        "dataset": "A (i.i.d.)",
        "param": "ell",
        "real": np.nan,  # not meaningful when sigma_f=0 (no signal)
        "learned": fit_uncorr["ell"],
        "abs_err": np.nan,
        "rel_err": np.nan,
    })
    rows.append({
        "dataset": "A (i.i.d.)",
        "param": "sigma_n",
        "real": float(sigma_iid_real),
        "learned": fit_uncorr["sigma_n"],
        "abs_err": abs(fit_uncorr["sigma_n"] - float(sigma_iid_real)),
        "rel_err": abs(fit_uncorr["sigma_n"] - float(sigma_iid_real)) / (abs(float(sigma_iid_real)) + 1e-12),
    })
    rows.append({
        "dataset": "A (i.i.d.)",
        "param": "log_marginal_likelihood",
        "real": np.nan,
        "learned": fit_uncorr["log_marginal_likelihood"],
        "abs_err": np.nan,
        "rel_err": np.nan,
    })

    # Dataset B
    rows.append({
        "dataset": "B (correlated)",
        "param": "sigma_f",
        "real": float(sigma_f_real),
        "learned": fit_corr["sigma_f"],
        "abs_err": abs(fit_corr["sigma_f"] - float(sigma_f_real)),
        "rel_err": abs(fit_corr["sigma_f"] - float(sigma_f_real)) / (abs(float(sigma_f_real)) + 1e-12),
    })
    rows.append({
        "dataset": "B (correlated)",
        "param": "ell",
        "real": float(ell_real),
        "learned": fit_corr["ell"],
        "abs_err": abs(fit_corr["ell"] - float(ell_real)),
        "rel_err": abs(fit_corr["ell"] - float(ell_real)) / (abs(float(ell_real)) + 1e-12),
    })
    rows.append({
        "dataset": "B (correlated)",
        "param": "sigma_n",
        "real": float(sigma_corr_real),
        "learned": fit_corr["sigma_n"],
        "abs_err": abs(fit_corr["sigma_n"] - float(sigma_corr_real)),
        "rel_err": abs(fit_corr["sigma_n"] - float(sigma_corr_real)) / (abs(float(sigma_corr_real)) + 1e-12),
    })
    rows.append({
        "dataset": "B (correlated)",
        "param": "log_marginal_likelihood",
        "real": np.nan,
        "learned": fit_corr["log_marginal_likelihood"],
        "abs_err": np.nan,
        "rel_err": np.nan,
    })

    df = pd.DataFrame(rows)

    # Nice formatting
    def fmt(x):
        if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
            return ""
        return f"{x:.6g}"

    df_fmt = df.copy()
    for col in ["real", "learned", "abs_err", "rel_err"]:
        df_fmt[col] = df_fmt[col].map(fmt)

    return df_fmt

df_table = make_real_vs_learned_table(
    sigma_iid_real=sigma_iid_real,
    sigma_corr_real=sigma_corr_real,
    sigma_f_real=sigma_f_real,
    ell_real=ell_real,
    fit_uncorr=fit_uncorr,
    fit_corr=fit_corr,
)

df_table

,dataset,param,real,learned,abs_err,rel_err
0,A (i.i.d.),sigma_f,0,0.192123,0.192123,
1,A (i.i.d.),ell,,2.95624,,
2,A (i.i.d.),sigma_n,0.5,0.39745,0.10255,0.205101
3,A (i.i.d.),log_marginal_likelihood,,-21.8973,,
4,B (correlated),sigma_f,1.5,0.721732,0.778268,0.518845
5,B (correlated),ell,1.2,1.12389,0.076115,0.0634291
6,B (correlated),sigma_n,0,1.15294e-05,1.15294e-05,1.15294e+07
7,B (correlated),log_marginal_likelihood,,199.758,,


In [10]:
def gp_predict_rbf(x_train, y_train, x_test, *, sigma_f, ell, sigma_n, jitter=1e-10):
    x_train = np.asarray(x_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.float64).reshape(-1)
    x_test  = np.asarray(x_test, dtype=np.float64)

    # Kernel matrices
    K   = K_rbf(x_train, sigma_f, ell)
    Ky  = K + (sigma_n**2 + jitter) * np.eye(len(x_train))

    K_s = (sigma_f**2) * np.exp(
        -0.5 * (x_test.reshape(-1,1) - x_train.reshape(1,-1))**2 / ell**2
    )
    K_ss = K_rbf(x_test, sigma_f, ell)

    # Cholesky
    c, lower = cho_factor(Ky, lower=True, check_finite=False)
    alpha = cho_solve((c, lower), y_train, check_finite=False)

    # Posterior mean
    mu = K_s @ alpha

    # Posterior covariance
    v = cho_solve((c, lower), K_s.T, check_finite=False)
    cov = K_ss - K_s @ v

    var = np.clip(np.diag(cov), 0.0, np.inf)

    return mu, var

In [11]:
x_star = np.linspace(x.min(), x.max(), 300)

mu_A, var_A = gp_predict_rbf(
    x, y_uncorr, x_star,
    sigma_f=fit_uncorr["sigma_f"],
    ell=fit_uncorr["ell"],
    sigma_n=fit_uncorr["sigma_n"],
)

mu_B, var_B = gp_predict_rbf(
    x, y_corr, x_star,
    sigma_f=fit_corr["sigma_f"],
    ell=fit_corr["ell"],
    sigma_n=fit_corr["sigma_n"],
)

In [13]:
import ROOT
import numpy as np
import uuid

def plot_gp_fit_root_inline(
    x_train, y_train,
    x_star, mu, var,
    *,
    title="GP posterior mean ± 2σ",
    canvas_name="c_gp",
    canvas_title="GP fit",
    width=1100,
    height=450,
    nsigma=2.0,
):
    # ROOT config: inline
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    # Avoid canvas name collisions
    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"

    # Ensure numpy float64 1D
    x_train = np.asarray(x_train, dtype=np.float64).ravel()
    y_train = np.asarray(y_train, dtype=np.float64).ravel()
    x_star  = np.asarray(x_star,  dtype=np.float64).ravel()
    mu      = np.asarray(mu,      dtype=np.float64).ravel()
    var     = np.asarray(var,     dtype=np.float64).ravel()

    std = np.sqrt(np.clip(var, 0.0, np.inf))

    c = ROOT.TCanvas(cname, canvas_title, width, height)

    # Keep references alive for JSROOT
    c._objs = []

    # --- Data points
    g_data = ROOT.TGraph(len(x_train), x_train, y_train)
    g_data.SetName(f"g_data_{uid}")
    g_data.SetMarkerStyle(20)
    g_data.SetMarkerSize(1.0)
    g_data.SetTitle(f"{title};x;y")
    c._objs.append(g_data)

    # --- Mean curve
    g_mu = ROOT.TGraph(len(x_star), x_star, mu)
    g_mu.SetName(f"g_mu_{uid}")
    g_mu.SetLineWidth(2)
    c._objs.append(g_mu)

    # --- Uncertainty band as filled polygon (TGraph)
    # Build polygon: go forward on upper, backward on lower
    y_up = mu + nsigma * std
    y_lo = mu - nsigma * std

    x_poly = np.concatenate([x_star, x_star[::-1]])
    y_poly = np.concatenate([y_up,   y_lo[::-1]])

    g_band = ROOT.TGraph(len(x_poly), x_poly.astype(np.float64), y_poly.astype(np.float64))
    g_band.SetName(f"g_band_{uid}")
    g_band.SetFillStyle(1001)
    g_band.SetFillColorAlpha(ROOT.kAzure - 9, 0.35)
    g_band.SetLineColorAlpha(ROOT.kAzure - 9, 0.0)
    c._objs.append(g_band)

    # Draw order
    g_data.Draw("AP")
    g_band.Draw("F SAME")  # filled polygon band
    g_mu.Draw("L SAME")
    g_data.Draw("P SAME")

    c.Modified()
    c.Update()

    # JSROOT inline render (same pattern as your working function)
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)

    return c

In [14]:
cA = plot_gp_fit_root_inline(
    x, y_uncorr, x_star, mu_A, var_A,
    title="Dataset A (i.i.d.) — GP posterior mean ± 2σ",
    canvas_name="cA"
)

cB = plot_gp_fit_root_inline(
    x, y_corr, x_star, mu_B, var_B,
    title="Dataset B (correlated) — GP posterior mean ± 2σ",
    canvas_name="cB"
)